# General Conference Talk Analysis

## Project Overview
This notebook analyzes talks from The Church of Jesus Christ of Latter-day Saints General Conference. We'll:
1. **Scrape talk data** from the official website
2. **Generate embeddings** using AI to convert text into numerical representations
3. **Visualize patterns** in the data using dimensionality reduction and interactive plots

This allows us to explore relationships between talks and identify thematic clusters without reading each one individually.

## Step 1: Import Required Libraries

Let's start by importing the tools we'll need for web scraping, data processing, and analysis.

In [ ]:
# Web scraping and data processing libraries
import requests                    # For downloading web pages
import bs4                         # For parsing HTML content
import pandas as pd               # For organizing data into tables
import numpy as np               # For numerical operations
from tqdm.auto import tqdm        # For progress bars during loops

# Machine learning and data visualization
import ollama                     # For generating text embeddings
import phate                      # For dimensionality reduction (converting high-dimensional data to 2D)
import plotly.express as px      # For interactive visualizations

c:\Users\tkerby2\Desktop\Teaching\Winter_2026\STAT_386\gc_analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2: Web Scraping - Extract Talk Data

We'll download General Conference talks from specific months and years. Here's what we're doing:

### Data Collection Strategy
- **Source**: The official Church website
- **Time Period**: April and October conferences from 2024-2025
- **Data Points**: Title, author, role, kicker (summary), and full text of each talk

### How It Works
1. Fetch the conference index page
2. Parse the HTML to find links to individual talks
3. For each link, download and extract the talk content
4. Store everything in a structured format (DataFrame)

In [ ]:
# Helper function to safely extract text from HTML elements
def extract_data(soup, tag, class_name):
    """
    Searches for an HTML element by tag and class, then extracts its text.
    
    Parameters:
    - soup: BeautifulSoup object containing parsed HTML
    - tag: HTML tag name (e.g., 'h1', 'p', 'div')
    - class_name: CSS class name (or None if searching by tag only)
    
    Returns: Text content if element found, None otherwise
    """
    element = soup.find(tag, class_=class_name)
    return element.get_text() if element else None

# Base URL for the Church website
base_url = "https://www.churchofjesuschrist.org/"

# Initialize a dictionary to store all our data
# Each key will become a column in our DataFrame
data_dict = {
    "title": [],      # Talk title
    "author": [],     # Speaker name
    "role": [],       # Speaker's position/responsibility
    "kicker": [],     # Summary/description
    "text": [],       # Full talk text
    "year": [],       # Conference year
    "month": []       # Conference month
}

# Loop through the conferences we want to collect (April and October of 2024-2025)
for year in [2024, 2025]:
    for month in ["04", "10"]:
        # Step 1: Download the conference index page
        response = requests.get(f"https://www.churchofjesuschrist.org/study/general-conference/{year}/{month}?lang=eng")

        # Step 2: Parse the HTML to find all talk links
        soup = bs4.BeautifulSoup(response.content, "html.parser")
        raw_urls = soup.find_all("a", class_="item-U_5Ca")  # Find all talk links
        urls = [a["href"] for a in raw_urls]
        
        # Skip the first link (it's usually not a talk)
        final_urls = urls[1:]
        print(f"Processing talks for {year}-{month}...")

        # Step 3: For each talk, download and extract the data
        for url in tqdm(final_urls):
            # Download the individual talk page
            talk = requests.get(base_url + url)
            talk_soup = bs4.BeautifulSoup(talk.content, "html.parser")
            
            # Extract each piece of information we need
            data_dict['title'].append(extract_data(talk_soup, 'h1', None))
            data_dict['author'].append(extract_data(talk_soup, 'p', 'author-name'))
            data_dict['role'].append(extract_data(talk_soup, 'p', 'author-role'))
            data_dict['kicker'].append(extract_data(talk_soup, 'p', 'kicker'))
            data_dict['text'].append(extract_data(talk_soup, 'div', 'body-block'))
            data_dict['year'].append(year)
            data_dict['month'].append(month)

# Step 4: Convert our dictionary into a pandas DataFrame for easier analysis
df = pd.DataFrame(data_dict)
df

Processing talks for 2024-04...


100%|██████████| 34/34 [00:23<00:00,  1.45it/s]


Processing talks for 2024-10...


100%|██████████| 35/35 [00:25<00:00,  1.37it/s]


Processing talks for 2025-04...


100%|██████████| 34/34 [00:21<00:00,  1.58it/s]


Processing talks for 2025-10...


100%|██████████| 35/35 [00:20<00:00,  1.69it/s]


,title,author,role,kicker,text,year,month
0,"Sustaining of General Authorities, Area Sevent...",Presented by President Dallin H. Oaks,First Counselor in the First Presidency,NaN,"\nBrothers and sisters, it will now be my priv...",2024,04
1,"Church Auditing Department Report, 2023",Presented by Jared B. Larson,"Managing Director, Church Auditing Department",To the First Presidency of The Church of Jesus...,"\nDear Brethren: Directed by revelation, as re...",2024,04
2,Motions of a Hidden Fire,By President Jeffrey R. Holland,Acting President of the Quorum of the Twelve A...,God hears every prayer we offer and responds t...,"\nBrothers and sisters, I have learned a painf...",2024,04
3,Put Ye On the Lord Jesus Christ,By Sister J. Anette Dennis,First Counselor in the Relief Society General ...,"Through honoring our covenants, we enable God ...","\nAs my two youngest children were growing, I ...",2024,04
4,Pillars and Rays,By Elder Alexander Dushku,Of the Seventy,We too can have our own pillar of light—one ra...,\nMy message is for those who worry about thei...,2024,04
...,...,...,...,...,...,...,...
133,Smiling Faces and Grateful Hearts,By Elder Carlos A. Godoy,Of the Seventy,The greatness of our Saints in Africa becomes ...,"\nA little over a year ago, I was released fro...",2025,10
134,Taking on the Name of Jesus Christ,By Elder Dale G. Renlund,Of the Quorum of the Twelve Apostles,The more we identify with and remember Jesus C...,"\nIn 2018, at the University of Utah, a specia...",2025,10
135,The Good News Recipe,By Elder John D. Amos,Of the Seventy,What might it look like to add more Jesus Chri...,\nIf you have ever visited my home state of Lo...,2025,10
136,The Book of Mormon—an Immeasurable Treasure on...,By Elder Ozani Farias,Of the Seventy,As we feast upon the words of Christ found in ...,\nCan you remember a moment when someone gave ...,2025,10


## Step 3: Generate Text Embeddings

### What Are Embeddings?
Embeddings convert text into vectors (lists of numbers) that capture the semantic meaning of the text. Words or texts with similar meanings will have similar embeddings.

### Why Embeddings?
- Computers can't understand raw text directly, but they can work with numbers
- We can measure similarity between talks by comparing their embeddings
- This is the foundation for clustering and visualization

### Our Approach
We use the `embeddinggemma` model via Ollama to generate embeddings for each talk's text. This creates a vector for each talk in a high-dimensional space.

In [ ]:
# For each talk, generate an embedding of its text using the embeddinggemma model
# This converts the natural language text into a vector of numbers that represents its meaning
df['embedding'] = df['text'].apply(lambda x: ollama.embed(model="embeddinggemma", input=x)["embeddings"][0])

## Step 4: Dimensionality Reduction & Visualization

### The Challenge: Too Many Dimensions
Each embedding has hundreds of dimensions, making it impossible to visualize directly. We can only see in 2D or 3D.

### The Solution: PHATE (Potential of Heat Diffusion for Affinity-based Transition Embedding)
PHATE is a dimensionality reduction technique that:
- Preserves the local structure of data (talks that are similar stay together)
- Reveals global structure and clusters
- Works better than t-SNE for exploring continuous structure

### What We're Creating
An interactive scatter plot where:
- **X and Y axes**: The 2D coordinates from PHATE
- **Color**: The year of the conference (2024 vs 2025)
- **Hover**: Click on points to see talk title, author, role, and date
- **Clustering**: Similar talks should appear near each other

In [ ]:
# Step 1: Apply PHATE dimensionality reduction to convert high-dimensional embeddings to 2D
phate_op = phate.PHATE(n_components=2)                    # Create PHATE reducer for 2D output
embeddings = np.stack(df['embedding'].values)             # Stack all embeddings into a matrix
phate_result = phate_op.fit_transform(embeddings)         # Apply PHATE to get 2D coordinates

# Step 2: Create an interactive scatter plot
fig = px.scatter(
    x=phate_result[:, 0],                                 # X-axis: first PHATE dimension
    y=phate_result[:, 1],                                 # Y-axis: second PHATE dimension
    color=df["year"],                                     # Color points by conference year
    hover_name=df["title"],                               # Show talk title on hover
    hover_data={                                          # Additional info to show on hover
        "author": df["author"],
        "role": df["role"],
        "year": df["year"],
        "month": df["month"]
    },
)

# Display the interactive visualization
fig.show()

Calculating PHATE...
  Running PHATE on 138 observations and 768 variables.
  Calculating graph and diffusion operator...
    Calculating PCA...
    Calculated PCA in 0.08 seconds.
    Calculating KNN search...
    Calculated KNN search in 1.95 seconds.
    Calculating affinities...
    Calculated affinities in 0.02 seconds.
  Calculated graph and diffusion operator in 2.06 seconds.
  Calculating optimal t...
    Automatically selected t = 12
  Calculated optimal t in 0.02 seconds.
  Calculating diffusion potential...
  Calculating metric MDS...
    SGD-MDS may not have converged: stress changed by -6.8% in final iterations. Consider increasing n_iter or adjusting learning_rate.
  Calculated metric MDS in 0.09 seconds.
Calculated PHATE in 2.18 seconds.
